# Historical research notebook — not a final Run All workflow
This is the initial openWakeWord notebook. The installed Italian V2 verifier was produced by subsequent interactive scripts archived beside it. See ../README.md and ../../../docs/WAKE_WORDS.md. Outputs were removed for privacy; a fresh end-to-end run of this notebook has NOT been validated.


# Nyra openWakeWord V1 — prototipo
Riutilizza il dataset sintetico V6 approvato. Due teste con soglie indipendenti. Verifica streaming obbligatoria prima della distribuzione; accuratezza umana non ancora misurata.


In [ ]:
"""Isolated Colab environment. Never modifies the V6 environment or production server."""
import sys,subprocess,os
from pathlib import Path

def run(*a):subprocess.run(list(a),check=True)
run(sys.executable,'-m','pip','install','-q','uv==0.9.7')
uv=[sys.executable,'-m','uv'];env=Path('/content/nyra_oww_env312');py=str(env/'bin/python')
if not env.exists():run(*uv,'venv','--python','3.12',str(env))
run(*uv,'pip','install','--python',py,'numpy==2.1.3','scipy==1.15.3','tensorflow[and-cuda]==2.19.1','keras==3.10.0','onnxruntime-gpu==1.23.2','scikit-learn==1.6.1','tqdm==4.67.1','requests==2.32.3')
for directory,url,commit in [('/content/nyra_oww_source','https://github.com/dscripka/openWakeWord.git','368c03716d1e92591906a84949bc477f3a834455'),('/content/nyra_pyoww_source','https://github.com/rhasspy/pyopen-wakeword.git','6bc5c5f5c9c71e46a723b6c9277b1d50f2ba13fd')]:
 if not Path(directory).exists():run('git','clone','--quiet',url,directory)
 run('git','-C',directory,'checkout','--quiet',commit)
run(*uv,'pip','install','--python',py,'--no-deps','-e','/content/nyra_oww_source')
run(py,'-c',"from openwakeword.utils import download_models; download_models(model_names=['__frontend_only__'])")
run(py,'-c',"import onnxruntime as ort; ort.preload_dlls(); import numpy as np; from openwakeword.utils import AudioFeatures; f=AudioFeatures(device='gpu',ncpu=2); assert f.onnx_execution_provider=='CUDAExecutionProvider',f.onnx_execution_provider; x=f.embed_clips(np.zeros((2,51200),dtype=np.int16),batch_size=2); print('FRONTEND_GPU_OK',x.shape); assert x.shape[0]==2 and x.shape[2]==96")
print('ENVIRONMENT_READY',py,flush=True)


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, os
drive.mount('/content/drive')
PYTHON='/content/nyra_oww_env312/bin/python'
WORKER=Path('/content/nyra_oww_worker.py')
WORKER.write_text("\"\"\"Nyra openWakeWord prototype: immutable V6 synthetic audio, frozen OWW frontend.\nThe new Keras head follows OWW's DNN/LayerNorm topology; it is not the microWakeWord trainer.\nNo real-room accuracy claims are inferred from synthetic evaluation.\n\"\"\"\nfrom pathlib import Path\nimport os,sys,json,hashlib,shutil,tarfile,collections,fcntl,time,importlib.util,argparse,gc\nROOT=Path('/content/nyra_oww_v1/oww-it-v1-001')\nSOURCE=Path('/content/drive/MyDrive/wakeword_training_nyra_it_v4/nyra-it-v6-004-continuous')\nBACKUP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001')\nOWW=Path('/content/nyra_oww_source');PYOWW=Path('/content/nyra_pyoww_source')\nPHRASES=('ehi_nira_it','nira_it');SEED=18473;FRONTEND_VERSION='oww-frozen-v2-exact-length'\nROOT.mkdir(parents=True,exist_ok=True);BACKUP.mkdir(parents=True,exist_ok=True)\ndef save(p,obj):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_suffix(p.suffix+'.partial');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False));tmp.replace(p)\ndef load(p):return json.loads(Path(p).read_text())\ndef sha(p):\n h=hashlib.sha256()\n with open(p,'rb') as f:\n  for b in iter(lambda:f.read(1<<20),b''):h.update(b)\n return h.hexdigest()\ndef guard(gib=8):\n assert shutil.disk_usage(ROOT).free>=gib*2**30,'Insufficient local disk: preserve existing files and stop'\ndef progress(stage,**kw):\n x=dict(stage=stage,updated=time.time(),**kw);save(ROOT/'progress.json',x);save(BACKUP/'progress.json',x);print(json.dumps(x),flush=True)\ndef old_worker():\n p=SOURCE.parent/'recovery_snapshots'/SOURCE.name/'nyra_v6_worker.py'\n spec=importlib.util.spec_from_file_location('approved_v6',p);m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m);return m\ndef extract():\n archive=SOURCE/'recovery_dataset_v1.tar.gz';assert archive.stat().st_size==2533330118\n if not (ROOT/'extraction_complete.json').exists():\n  guard(25)\n  with tarfile.open(archive) as t:\n   for m in t.getmembers():\n    parts=Path(m.name).parts\n    assert parts and parts[0] in ('audio','dataset.json') and '..' not in parts and not m.issym() and not m.islnk(),m.name\n   for m in t.getmembers():\n    if not (ROOT/m.name).exists():t.extract(m,ROOT,filter='data')\n d=load(ROOT/'dataset.json');assert len({r['id'] for r in d['rows']})==len(d['rows'])\n assert all((ROOT/r['path']).is_file() for r in d['rows'])\n save(ROOT/'extraction_complete.json',dict(rows=len(d['rows']),dataset_sha256=sha(ROOT/'dataset.json')))\n save(BACKUP/'source_provenance.json',dict(source=str(SOURCE),archive_bytes=archive.stat().st_size,dataset_sha256=sha(ROOT/'dataset.json'),all_audio_synthetic=True,source_splits_preserved=True,feature_policy=FRONTEND_VERSION,oww_commit=os.popen(f'git -C {OWW} rev-parse HEAD').read().strip(),pyoww_commit=os.popen(f'git -C {PYOWW} rev-parse HEAD').read().strip(),limitations=['Frozen speech embedding is primarily English-pretrained; Italian generalization requires physical tests.','Validation/test share synthetic voice identities with training; not a human-speaker generalization test.']))\n progress('extract_complete',rows=len(d['rows']))\ndef features():\n import numpy as np\n from scipy.io.wavfile import read\n import onnxruntime as ort\n ort.preload_dlls()\n from openwakeword.utils import AudioFeatures\n old=old_worker();F=AudioFeatures(device='gpu',ncpu=2)\n assert F.onnx_execution_provider=='CUDAExecutionProvider',F.onnx_execution_provider\n rows=load(ROOT/'dataset.json')['rows'];manifest=[];pending=[];count=0\n def flush():\n  nonlocal pending,count\n  if not pending:return\n  # Batch only exactly equal lengths: no padded tails in scoring or training.\n  assert len({len(a) for a,_ in pending})==1\n  audio=np.stack([a for a,_ in pending])\n  emb=F.embed_clips(audio,batch_size=min(64,len(audio)),ncpu=2)\n  for i,(_,meta) in enumerate(pending):\n   valid_mels=round(meta['duration']*16000)//160-3\n   valid_embeddings=(valid_mels-76)//8+1\n   x=emb[i][:valid_embeddings].astype(np.float32);assert x.ndim==2 and x.shape[1]==96 and len(x)>=16 and np.isfinite(x).all(),x.shape\n   if meta['split']=='training':x=x[-16:]\n   dst=ROOT/meta['feature'];dst.parent.mkdir(parents=True,exist_ok=True);np.save(dst,x)\n   count+=1\n  pending=[]\n def add(a,meta):\n  manifest.append(meta)\n  if (ROOT/meta['feature']).exists():return\n  # Official batch helper uses ceil for mel allocation, while the model uses floor.\n  # Pad universally to a 10 ms boundary, then retain only original complete frames.\n  if len(a)%160:a=np.pad(a,(0,160-len(a)%160))\n  if pending and len(pending[0][0])!=len(a):flush()\n  pending.append((a,meta))\n  if len(pending)>=32 or sum(len(x) for x,_ in pending)>32*128000:flush()\n for ri,r in enumerate(rows):\n  sr,a=read(ROOT/r['path']);assert sr==16000 and a.dtype==np.int16\n  assert sha(ROOT/r['path'])==r['sha256'],'WAV hash mismatch: '+r['path']\n  seed=SEED+r['id']*7919;rng=np.random.default_rng(seed)\n  base=dict(source_id=r['id'],kind=r['kind'],phrase=r['phrase'],split=r['split'],family=r['family'],variant=r.get('variant','unknown'),utterance_duration=r['duration'])\n  if r['split']=='training':\n   b=a.astype(np.float32)/32768\n   if r['kind']!='noise':b=old.acoustic(b,rng.choice(old.ACOUSTIC_PROFILES),seed)\n   b=b*10**float(rng.uniform(-18,0)/20)\n   n=51200\n   if len(b)>n:\n    start=int(rng.integers(0,len(b)-n+1));b=b[start:start+n]\n   tail=float(rng.uniform(.02,.25)) if len(b)<n-4000 else 0.\n   lead=max(0,(n-len(b))/16000-tail)\n   b=old.continuous_context(b,seed,pre_s=lead,tail_s=tail)\n   if len(b)<n:b=np.pad(b,(0,n-len(b)))\n   meta=dict(base,condition='training',duration=3.2,start=0.,end=3.2,feature=f'features/training/{r[\"id\"]}.npy')\n   add(np.clip(b[:n]*32767,-32768,32767).astype(np.int16),meta)\n  else:\n   for condition,b in old.evaluation_variants(a,r['kind'],seed):\n    meta=dict(base,condition=condition,duration=len(b)/16000,start=4.,end=4.+r['duration'],feature=f'features/{r[\"split\"]}/{r[\"id\"]}_{condition}.npy')\n    add(np.clip(b*32767,-32768,32767).astype(np.int16),meta)\n  if ri%100==0:\n   flush();guard();progress('features',source_rows_done=ri,total_source_rows=len(rows),new_feature_files=count)\n flush();save(ROOT/'features.json',dict(policy=FRONTEND_VERSION,dataset_sha256=sha(ROOT/'dataset.json'),rows=manifest));save(BACKUP/'features_manifest.json',dict(policy=FRONTEND_VERSION,rows=len(manifest),dataset_sha256=sha(ROOT/'dataset.json')));progress('features_complete',cases=len(manifest))\ndef tf_init():\n import tensorflow as tf\n tf.config.threading.set_intra_op_parallelism_threads(2);tf.config.threading.set_inter_op_parallelism_threads(2)\n for g in tf.config.list_physical_devices('GPU'):tf.config.experimental.set_memory_growth(g,True)\n tf.keras.utils.set_random_seed(SEED);return tf\ndef train():\n import numpy as np\n tf=tf_init();rows=load(ROOT/'features.json')['rows']\n for phrase in PHRASES:\n  modeldir=ROOT/'models'/phrase;modeldir.mkdir(parents=True,exist_ok=True);backup=BACKUP/'models'/phrase;backup.mkdir(parents=True,exist_ok=True)\n  for name in ('checkpoint.keras','state.json','best.keras','complete.json'):\n   if not (modeldir/name).exists() and (backup/name).exists():shutil.copy2(backup/name,modeldir/name)\n  if (modeldir/'complete.json').exists():\n   assert (modeldir/'best.keras').exists()\n   continue\n  selected=[r for r in rows if r['split']=='training' and (r['kind']!='positive' or r['phrase']==phrase)]\n  X=np.stack([np.load(ROOT/r['feature']) for r in selected]);y=np.array([r['kind']=='positive' for r in selected],np.float32)\n  pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0)\n  vr=[r for r in rows if r['split']=='validation' and r['kind']!='ambient' and (r['kind']!='positive' or r['phrase']==phrase)]\n  rng=np.random.default_rng(SEED);chosen=[];vx=[];vy=[]\n  for positive in (True,False):\n   candidates=[j for j,r in enumerate(vr) if (r['kind']=='positive')==positive]\n   chosen.extend(rng.choice(candidates,min(1500,len(candidates)),replace=False))\n  for j in chosen:\n   r=vr[j];a=np.load(ROOT/r['feature']);end=min(len(a),max(16,round((r['end']-.775)/.08)+1));vx.append(a[end-16:end]);vy.append(r['kind']=='positive')\n  vx=np.stack(vx);vy=np.array(vy,np.float32)\n  checkpoint=modeldir/'checkpoint.keras';statefile=modeldir/'state.json'\n  if not checkpoint.exists() and (backup/'checkpoint.keras').exists():shutil.copy2(backup/'checkpoint.keras',checkpoint);shutil.copy2(backup/'state.json',statefile)\n  if checkpoint.exists():m=tf.keras.models.load_model(checkpoint);state=load(statefile)\n  else:\n   inp=tf.keras.Input((16,96));x=tf.keras.layers.Flatten()(inp)\n   for units in (128,128):x=tf.keras.layers.Dense(units,kernel_regularizer=tf.keras.regularizers.L2(1e-5))(x);x=tf.keras.layers.LayerNormalization()(x);x=tf.keras.layers.ReLU()(x)\n   out=tf.keras.layers.Dense(1,activation='sigmoid')(x);m=tf.keras.Model(inp,out,name=phrase);m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='binary_crossentropy');state=dict(step=0,best_loss=1e9)\n  for step in range(state['step'],2000):\n   rng=np.random.default_rng(SEED+step);idx=np.r_[rng.choice(pos,128),rng.choice(neg,128)];loss=float(m.train_on_batch(X[idx],y[idx]))\n   if (step+1)%250==0:\n    vp=m(vx,training=False);vl=float(tf.reduce_mean(tf.keras.losses.binary_crossentropy(vy[:,None],vp)));state.update(step=step+1)\n    if vl<state['best_loss']:state['best_loss']=vl;m.save(modeldir/'best.keras');shutil.copy2(modeldir/'best.keras',backup/'best.keras')\n    m.save(checkpoint);save(statefile,state)\n    for name in ('checkpoint.keras','state.json'):shutil.copy2(modeldir/name,backup/name)\n    progress('train',phrase=phrase,step=step+1,train_loss=loss,validation_loss=vl)\n  save(modeldir/'complete.json',state);shutil.copy2(modeldir/'complete.json',backup/'complete.json');del m,X,vx;tf.keras.backend.clear_session();gc.collect()\n progress('train_complete')\ndef score(split):\n import numpy as np,gzip\n tf=tf_init();models=[tf.keras.models.load_model(ROOT/'models'/p/'best.keras',compile=False) for p in PHRASES]\n @tf.function(input_signature=[tf.TensorSpec([None,16,96],tf.float32)])\n def pred(x):return tf.concat([m(x,training=False) for m in models],axis=1)\n rows=[r for r in load(ROOT/'features.json')['rows'] if r['split']==split];results=[]\n for i,r in enumerate(rows):\n  x=np.load(ROOT/r['feature']);windows=np.lib.stride_tricks.sliding_window_view(x,(16,96))[:,0];p=pred(np.ascontiguousarray(windows)).numpy()\n  results.append(dict(r,probs={phrase:p[:,j].tolist() for j,phrase in enumerate(PHRASES)}))\n  if i%500==0:progress('score_'+split,done=i,total=len(rows))\n path=ROOT/f'scores_{split}.json.gz'\n with gzip.open(path,'wt') as f:json.dump(results,f)\n shutil.copy2(path,BACKUP/path.name);progress('score_'+split+'_complete',cases=len(results))\ndef events(probs,cutoff,refractory=2.):\n out=[]\n for i,p in enumerate(probs):\n  t=2.+i*.08\n  if t>=3.2 and p>cutoff and (not out or t-out[-1]>=refractory):out.append(t)\n return out\ndef metrics(rows,phrase,cutoff):\n pos=[r for r in rows if r['kind']=='positive' and r['phrase']==phrase];ambient=[r for r in rows if r['kind']=='ambient'];hard=[r for r in rows if r['kind']=='hard'];hours=sum(r['duration'] for r in ambient)/3600\n def hit(r):return any(r['start']<=t<=r['end']+1 for t in events(r['probs'][phrase],cutoff))\n conditions={c:sum(hit(r) for r in pos if r['condition']==c)/sum(r['condition']==c for r in pos) for c in sorted({r['condition'] for r in pos})}\n return dict(cutoff=cutoff,recall=sum(map(hit,pos))/len(pos),min_condition=min(conditions.values()),recall_by_condition=conditions,faph=sum(len(events(r['probs'][phrase],cutoff)) for r in ambient)/hours,hard_fpr=sum(bool(events(r['probs'][phrase],cutoff)) for r in hard)/len(hard))\ndef calibrate():\n import gzip\n with gzip.open(ROOT/'scores_validation.json.gz','rt') as f:rows=json.load(f)\n result={}\n for p in PHRASES:\n  grid=[metrics(rows,p,c) for c in (.05,.1,.15,.2,.25,.3,.4,.5,.6,.7,.8,.85,.9,.95,.975,.99,.995,.999)]\n  good=[r for r in grid if r['recall']>=.99 and r['min_condition']>=.95]\n  result[p]=dict(selected=min(good,key=lambda r:(r['faph'],r['hard_fpr'],-r['recall'])) if good else None,grid=grid)\n save(ROOT/'operating_points.json',result);save(BACKUP/'operating_points.json',result);progress('calibrate_complete',selected={p:r['selected'] for p,r in result.items()})\ndef export():\n import numpy as np,gzip\n tf=tf_init();ops=load(ROOT/'operating_points.json');assert all(ops[p]['selected'] for p in PHRASES),'No valid positive-recall operating point. Keep Okay Nabu installed.'\n with gzip.open(ROOT/'scores_testing.json.gz','rt') as f:rows=json.load(f)\n report={p:metrics(rows,p,ops[p]['selected']['cutoff']) for p in PHRASES}\n assert all(report[p]['recall']>=.97 and report[p]['min_condition']>=.9 for p in PHRASES),'Held-out synthetic recall gate failed; do not deploy.'\n out=ROOT/'export';out.mkdir(exist_ok=True);inp=tf.keras.Input(batch_shape=(1,16,96),dtype=tf.float32);scaled=[]\n for p in PHRASES:\n  m=tf.keras.models.load_model(ROOT/'models'/p/'best.keras',compile=False);m.trainable=False\n  # Each independent cutoff maps to the service's shared >0.5 threshold.\n  score=tf.keras.layers.Rescaling(1/(2*ops[p]['selected']['cutoff']))(m(inp));scaled.append(score)\n combo=tf.keras.Model(inp,tf.keras.layers.Maximum()(scaled),name='nira_ehi_nira')\n combo.export(str(ROOT/'saved_model'));conv=tf.lite.TFLiteConverter.from_saved_model(str(ROOT/'saved_model'));conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS];data=conv.convert();path=out/'nira_ehi_nira.tflite';path.write_bytes(data)\n it=tf.lite.Interpreter(model_content=data);it.allocate_tensors();assert it.get_input_details()[0]['shape'].tolist()==[1,16,96];assert it.get_output_details()[0]['shape'].tolist()==[1,1]\n for _ in range(10):\n  x=np.random.default_rng(_).normal(size=(1,16,96)).astype(np.float32);it.set_tensor(it.get_input_details()[0]['index'],x);it.invoke();assert np.allclose(it.get_tensor(it.get_output_details()[0]['index']),combo(x).numpy(),atol=1e-4)\n # Actual combined FAPH, not the sum of per-model rates.\n ambient=[r for r in rows if r['kind']=='ambient'];hours=sum(r['duration'] for r in ambient)/3600\n def combined(r):return [max(r['probs'][p][i]/(2*ops[p]['selected']['cutoff']) for p in PHRASES) for i in range(len(r['probs'][PHRASES[0]]))]\n report['combined_faph']=sum(len(events(combined(r),.5)) for r in ambient)/hours\n report.update(all_audio_synthetic=True,human_custom_word_test_performed=False,exact_pyopen_streaming_test_pending=True,batch_frontend_timing_approximation=True,threshold_selection='validation_only',model_sha256=sha(path),head_trainer='Keras adaptation of OWW DNN topology; frozen official frontend',operating_points={p:ops[p]['selected'] for p in PHRASES})\n save(out/'evaluation.json',report);save(out/'provenance.json',load(BACKUP/'source_provenance.json'));shutil.copytree(out,BACKUP/'export',dirs_exist_ok=True);zipfile=shutil.make_archive(str(ROOT/'nyra_openwakeword_v1_candidate'),'zip',out);shutil.copy2(zipfile,BACKUP/Path(zipfile).name);progress('candidate_exported_streaming_check_required',model_sha256=sha(path),metrics=report)\nif __name__=='__main__':\n phase=sys.argv[1]\n with open(ROOT/'worker.lock','w') as lock:\n  fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n  if phase=='extract':extract()\n  elif phase=='features':features()\n  elif phase=='train':train()\n  elif phase in ('score_validation','score_testing'):score(phase.removeprefix('score_'))\n  elif phase=='calibrate':calibrate()\n  elif phase=='export':export()\n  else:raise ValueError(phase)\n")
subprocess.run([PYTHON,'-u',str(WORKER),'extract'],check=True)


In [ ]:
import subprocess
probe=subprocess.run([PYTHON,'-c',"import onnxruntime as ort; ort.preload_dlls(); import numpy as np; from openwakeword.utils import AudioFeatures; f=AudioFeatures(device='gpu',ncpu=2); assert f.onnx_execution_provider=='CUDAExecutionProvider';\nfor n in (51199,51200,51201,89123,160000):\n a=np.zeros((2,((n+159)//160)*160),np.int16); x=f.embed_clips(a,batch_size=2); valid=((n//160-3)-76)//8+1; assert x[:,:valid].shape==(2,valid,96); print('BOUNDARY_OK',n,x.shape,valid,flush=True)"],capture_output=True,text=True)
print(probe.stdout,probe.stderr)
assert probe.returncode==0,'Test di regressione feature fallito'
import subprocess, time
from pathlib import Path
ROOT=Path('/content/nyra_oww_v1/oww-it-v1-001')
assert (ROOT/'extraction_complete.json').exists(), 'Recupero non concluso: non avviare altre fasi'
def stage(phase):
    print('FASE',phase,flush=True)
    with (ROOT/(phase+'.log')).open('a') as log:
        with subprocess.Popen([PYTHON,'-u',str(WORKER),phase],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
            for line in proc.stdout:
                log.write(line);log.flush();print(line,end='',flush=True)
            code=proc.wait()
    if code: raise RuntimeError(f'Fase {phase} interrotta, codice {code}; dataset e checkpoint conservati')
for phase in ('features','train','score_validation','calibrate','score_testing','export'):
    stage(phase)
print('CANDIDATO PRONTO: verifica streaming esatta ancora obbligatoria, nessuna installazione automatica da questa cella.')


In [ ]:
assert (ROOT/'export/nira_ehi_nira.tflite').is_file(), 'Candidato non disponibile: controllare la fase precedente'
CHECK=Path('/content/nyra_exact_stream_check.py')
CHECK.write_text("\"\"\"Evaluate candidate using the production-style pyopen_wakeword TFLite streaming backend.\nThis is a synthetic compatibility/recall test, not a physical-room accuracy test.\n\"\"\"\nfrom pathlib import Path\nimport sys,json,importlib.util,time,collections,hashlib\nimport numpy as np\nfrom scipy.io.wavfile import read\nsys.path.insert(0,'/content/nyra_pyoww_source')\nfrom pyopen_wakeword import OpenWakeWord,OpenWakeWordFeatures\nROOT=Path('/content/nyra_oww_v1/oww-it-v1-001');BACKUP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001')\nspec=importlib.util.spec_from_file_location('worker','/content/nyra_oww_worker.py');w=importlib.util.module_from_spec(spec);spec.loader.exec_module(w)\nold=w.old_worker();head=OpenWakeWord.from_model(ROOT/'export/nira_ehi_nira.tflite');front=OpenWakeWordFeatures.from_builtin();rows=w.load(ROOT/'dataset.json')['rows']\nassert w.sha(ROOT/'export/nira_ehi_nira.tflite')==w.load(ROOT/'export/evaluation.json')['model_sha256']\nrng=np.random.default_rng(91837);selected=[]\nfor phrase in w.PHRASES:\n candidates=[r for r in rows if r['split']=='testing' and r['kind']=='positive' and r['phrase']==phrase]\n selected.extend(candidates[i] for i in rng.choice(len(candidates),50,replace=False))\n# Use all held-out ambient and hard negatives, not just favorable short clips.\nselected.extend(r for r in rows if r['split']=='testing' and r['kind'] in ('ambient','hard','speech'))\nresults=[];started=time.time()\nfor index,r in enumerate(selected):\n sr,a=read(ROOT/r['path']);assert sr==16000 and a.dtype==np.int16;assert w.sha(ROOT/r['path'])==r['sha256']\n for condition,b in old.evaluation_variants(a,r['kind'],w.SEED+r['id']*7919):\n  front.reset();head.reset();pcm=np.clip(b*32767,-32768,32767).astype('<i2');detections=[];max_score=0.\n  for offset in range(0,len(pcm),1280):\n   chunk=pcm[offset:offset+1280]\n   if len(chunk)<1280:chunk=np.pad(chunk,(0,1280-len(chunk)))\n   timestamp=min((offset+1280)/16000,len(pcm)/16000)\n   for emb in front.process_streaming(chunk.tobytes()):\n    for p in head.process_streaming(emb):\n     assert np.isfinite(p);max_score=max(max_score,float(p))\n     if timestamp>=3.2 and p>.5 and (not detections or timestamp-detections[-1]>=2):detections.append(timestamp)\n  results.append(dict(source_id=r['id'],phrase=r['phrase'],kind=r['kind'],condition=condition,duration=len(pcm)/16000,detections=detections,hit=any(4<=t<=4+r['duration']+1 for t in detections),max_scaled_score=max_score))\n if index%25==0:\n  w.save(ROOT/'exact_stream_partial.json',dict(source_rows_done=index,total=len(selected),cases=len(results),elapsed=time.time()-started));print('EXACT_STREAM',index,len(selected),len(results),flush=True)\npositive=[r for r in results if r['kind']=='positive'];ambient=[r for r in results if r['kind']=='ambient'];hard=[r for r in results if r['kind']=='hard'];hours=sum(r['duration'] for r in ambient)/3600\nrecall={p:sum(r['hit'] for r in positive if r['phrase']==p)/sum(r['phrase']==p for r in positive) for p in w.PHRASES}\nby_condition={p:{c:sum(r['hit'] for r in positive if r['phrase']==p and r['condition']==c)/sum(r['phrase']==p and r['condition']==c for r in positive) for c in sorted({r['condition'] for r in positive})} for p in w.PHRASES}\nreport=dict(model_sha256=w.sha(ROOT/'export/nira_ehi_nira.tflite'),backend_commit='6bc5c5f5c9c71e46a723b6c9277b1d50f2ba13fd',recall=recall,recall_by_condition=by_condition,ambient_hours=hours,faph=sum(len(r['detections']) for r in ambient)/hours,hard_negative_fpr=sum(bool(r['detections']) for r in hard)/len(hard),all_audio_synthetic=True,human_test=False,thresholds_selected_on='batch_validation_only',cases=len(results),elapsed_seconds=time.time()-started)\nreport['synthetic_positive_gate_passed']=all(x>=.97 for x in recall.values()) and all(min(v.values())>=.9 for v in by_condition.values())\nw.save(ROOT/'export/exact_stream_evaluation.json',report);w.save(BACKUP/'export/exact_stream_evaluation.json',report);w.save(ROOT/'exact_stream_cases.json',results)\nprint(json.dumps(report,indent=2),flush=True)\nassert report['synthetic_positive_gate_passed'],'Exact streaming positive recall failed: do not deploy. Revisit features/calibration with validation only.'\n")
with subprocess.Popen([PYTHON,'-u',str(CHECK)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
    for line in proc.stdout: print(line,end='',flush=True)
    assert proc.wait()==0, 'Controllo streaming non superato: non installare'


In [ ]:
from pathlib import Path
import json, collections, subprocess
ROOT=Path('/content/nyra_oww_v1/oww-it-v1-001')
ops=json.loads((ROOT/'operating_points.json').read_text())
for phrase, info in ops.items():
 print('CALIBRAZIONE',phrase)
 for r in info['grid']: print({k:r[k] for k in ('cutoff','recall','min_condition','faph','hard_fpr')})
 print('STATO TRAIN',json.loads((ROOT/'models'/phrase/'complete.json').read_text()))
print('PROCESSI',subprocess.run(['pgrep','-af','nyra_oww_worker.py'],capture_output=True,text=True).stdout)


In [ ]:
import gzip, json, numpy as np
with gzip.open(ROOT/'scores_validation.json.gz','rt') as f: scores=json.load(f)
for phrase in ('ehi_nira_it','nira_it'):
 rs=[r for r in scores if r['kind']=='positive' and r['phrase']==phrase]
 for cut in (.5,.9):
  counts=dict(cases=len(rs),crosses_in_phrase=0,false_before_phrase=0,all_below=0)
  for r in rs:
   p=np.asarray(r['probs'][phrase]); t=2+np.arange(len(p))*.08
   counts['crosses_in_phrase']+=int(np.any((t>=r['start'])&(t<=r['end']+1)&(p>cut)))
   counts['false_before_phrase']+=int(np.any((t>=3.2)&(t<r['start'])&(p>cut)))
   counts['all_below']+=int(not np.any(p>cut))
  print(phrase,cut,counts)
print('SOURCE NEGATIVE DURATIONS',[(k,[(r['id'],round(r['duration'],2)) for r in json.loads((ROOT/'dataset.json').read_text())['rows'] if r['split']=='training' and r['kind']==k][:3]) for k in ('continuous','speech','hard','noise')])


In [ ]:
from pathlib import Path
import subprocess, shutil
TUNE=Path('/content/nyra_tune_background_v2.py')
TUNE.write_text("\"\"\"Add quiet-background coverage and mine training-only hard negatives; preserve V1.\"\"\"\nfrom pathlib import Path\nimport sys,json,shutil,importlib.util,collections,time,fcntl\nimport numpy as np\nspec=importlib.util.spec_from_file_location('w','/content/nyra_oww_worker.py');w=importlib.util.module_from_spec(spec);spec.loader.exec_module(w)\nBASE=w.ROOT;NEW=BASE/'tuning_v2';BKP=w.BACKUP/'tuning_v2';NEW.mkdir(exist_ok=True);BKP.mkdir(exist_ok=True)\nlock=open(NEW/'tuning.lock','w');fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\ntf=w.tf_init()\nimport onnxruntime as ort\nort.preload_dlls()\nfrom openwakeword.utils import AudioFeatures\nfrom scipy.io.wavfile import read\nF=AudioFeatures(device='gpu',ncpu=2);assert F.onnx_execution_provider=='CUDAExecutionProvider'\nold=w.old_worker();oldmodels=[tf.keras.models.load_model(BASE/'models'/p/'best.keras',compile=False) for p in w.PHRASES]\n@tf.function(input_signature=[tf.TensorSpec([None,16,96],tf.float32)])\ndef oldscore(x):return tf.reduce_max(tf.concat([m(x,training=False) for m in oldmodels],axis=1),axis=1)\ndef embed(a):\n n=len(a)\n if n%160:a=np.pad(a,(0,160-n%160))\n x=F.embed_clips(a[None],batch_size=1,ncpu=2)[0];return x[:((n//160-3)-76)//8+1].astype(np.float32)\ndef windows(x):return np.ascontiguousarray(np.lib.stride_tricks.sliding_window_view(x,(16,96))[:,0])\nrows=w.load(BASE/'dataset.json')['rows'];mining=[r for r in rows if r['split']=='training' and r['kind'] in ('hard','continuous')]\npath=NEW/'mined_negatives.npy';partial=NEW/'mined_negatives.partial.npy';statepath=NEW/'mining_state.json'\nif not path.exists():\n shape=(len(mining)*4,16,96)\n bank=np.lib.format.open_memmap(partial,mode='r+' if partial.exists() else 'w+',dtype=np.float32,shape=shape)\n done=w.load(statepath)['done'] if statepath.exists() else 0\n for i in range(done,len(mining)):\n  r=mining[i];sr,a=read(BASE/r['path']);assert sr==16000 and w.sha(BASE/r['path'])==r['sha256']\n  seed=w.SEED+r['id']*7919;a=a.astype(np.float32)/32768\n  a=old.acoustic(a,old.ACOUSTIC_PROFILES[i%len(old.ACOUSTIC_PROFILES)],seed)\n  a=old.continuous_context(a,seed,pre_s=4.,tail_s=1.)\n  x=windows(embed(np.clip(a*32767,-32768,32767).astype(np.int16)));s=oldscore(x).numpy();idx=np.argsort(s)[-4:];assert len(idx)==4\n  bank[i*4:i*4+4]=x[idx]\n  if (i+1)%100==0:\n   bank.flush();w.save(statepath,dict(done=i+1,total=len(mining)));print('TRAIN_ONLY_MINING',i+1,len(mining),flush=True)\n bank.flush();del bank;partial.replace(path);w.save(statepath,dict(done=len(mining),total=len(mining)))\nquietpath=NEW/'quiet_background.npy'\nif not quietpath.exists():\n quiet=[]\n for start in range(0,4096,32):\n  audio=[]\n  for i in range(start,start+32):\n   db=(-80,-75,-70,-65,-60,-55,-50,-45)[i%8]\n   a=old.continuous_context(np.empty(0,np.float32),w.SEED+100000+i,pre_s=3.2,tail_s=0.,floor_dbfs=db)\n   audio.append(np.clip(a*32767,-32768,32767).astype(np.int16))\n  quiet.extend(F.embed_clips(np.stack(audio),batch_size=32,ncpu=2)[:,-16:])\n np.save(quietpath,np.stack(quiet).astype(np.float32))\ndel F,oldmodels;tf.keras.backend.clear_session()\n# New model directory and immutable absolute references to existing features/audio.\nfeatures=w.load(BASE/'features.json')\nfor r in features['rows']:r['feature']=str(BASE/r['feature'])\nw.save(NEW/'features.json',features)\ndataset=w.load(BASE/'dataset.json')\nfor r in dataset['rows']:r['path']=str(BASE/r['path'])\nw.save(NEW/'dataset.json',dataset)\nprovenance=w.load(w.BACKUP/'source_provenance.json');provenance.update(tuning='v2: training-only hard-negative mining and quiet backgrounds; V1 preserved',source_model_root=str(BASE/'models'),quiet_floor_dbfs=[-80,-75,-70,-65,-60,-55,-50,-45]);w.save(BKP/'source_provenance.json',provenance)\nmined=np.load(path,mmap_mode='r');quiet=np.load(quietpath,mmap_mode='r')\nfor phrase in w.PHRASES:\n dest=NEW/'models'/phrase;dest.mkdir(parents=True,exist_ok=True);backup=BKP/'models'/phrase;backup.mkdir(parents=True,exist_ok=True)\n for name in ('checkpoint.keras','state.json','best.keras','complete.json'):\n  if not (dest/name).exists() and (backup/name).exists():shutil.copy2(backup/name,dest/name)\n if (dest/'complete.json').exists():continue\n rr=[r for r in features['rows'] if r['split']=='training' and (r['kind']!='positive' or r['phrase']==phrase)]\n X=np.stack([np.load(r['feature']) for r in rr]);y=np.array([r['kind']=='positive' for r in rr]);pos=np.flatnonzero(y);neg=np.flatnonzero(~y)\n vr=[r for r in features['rows'] if r['split']=='validation' and r['kind']!='ambient' and (r['kind']!='positive' or r['phrase']==phrase)]\n rng=np.random.default_rng(w.SEED);vx=[];vy=[]\n for positive,number in ((True,1500),(False,750)):\n  candidates=[r for r in vr if (r['kind']=='positive')==positive]\n  for j in rng.choice(len(candidates),min(number,len(candidates)),replace=False):\n   r=candidates[j];a=np.load(r['feature']);end=min(len(a),max(16,round((r['end']-.775)/.08)+1));vx.append(a[end-16:end]);vy.append(float(positive))\n # Validation background prefixes are separate from generated training backgrounds.\n candidates=[r for r in vr if r['kind']=='positive']\n for j in rng.choice(len(candidates),750,replace=False):vx.append(np.load(candidates[j]['feature'])[15:31]);vy.append(0.)\n vx=np.stack(vx);vy=np.asarray(vy,np.float32)[:,None]\n if (dest/'checkpoint.keras').exists():m=tf.keras.models.load_model(dest/'checkpoint.keras');st=w.load(dest/'state.json')\n else:\n  m=tf.keras.models.load_model(BASE/'models'/phrase/'best.keras');m.optimizer.learning_rate.assign(3e-4);st=dict(step=0,best_loss=1e9,source='V1 best; optimizer retained, learning rate lowered to 0.0003')\n for step in range(st['step'],4000):\n  rng=np.random.default_rng(w.SEED+200000+step);xp=X[rng.choice(pos,128)];xn=np.concatenate([X[rng.choice(neg,48)],mined[rng.choice(len(mined),48)],quiet[rng.choice(len(quiet),32)]])\n  loss=float(m.train_on_batch(np.concatenate([xp,xn]),np.r_[np.ones(128),np.zeros(128)].astype(np.float32)))\n  if (step+1)%250==0:\n   vl=float(tf.reduce_mean(tf.keras.losses.binary_crossentropy(vy,m(vx,training=False))))\n   st['step']=step+1\n   if vl<st['best_loss']:st['best_loss']=vl;m.save(dest/'best.keras');shutil.copy2(dest/'best.keras',backup/'best.keras')\n   m.save(dest/'checkpoint.keras');w.save(dest/'state.json',st)\n   for name in ('checkpoint.keras','state.json'):shutil.copy2(dest/name,backup/name)\n   print('BACKGROUND_TUNING',phrase,step+1,loss,vl,flush=True)\n w.save(dest/'complete.json',st);shutil.copy2(dest/'complete.json',backup/'complete.json');del m,X,vx;tf.keras.backend.clear_session()\n# Reuse audited scoring/export code with a new root, leaving V1 scores and weights intact.\nsource=Path('/content/nyra_oww_worker.py').read_text().replace(\"ROOT=Path('/content/nyra_oww_v1/oww-it-v1-001')\",f'ROOT=Path({str(NEW)!r})').replace(\"BACKUP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001')\",f'BACKUP=Path({str(BKP)!r})')\n(NEW/'worker.py').write_text(source);shutil.copy2(NEW/'worker.py',BKP/'worker.py')\nprint('BACKGROUND_TUNING_COMPLETE',NEW,flush=True)\n")
BKP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001/tuning_v2')
BKP.mkdir(parents=True,exist_ok=True)
shutil.copy2(TUNE,BKP/TUNE.name)
def logged(args):
 with subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
  for line in proc.stdout: print(line,end='',flush=True)
  code=proc.wait()
 if code: raise RuntimeError(f'Fase interrotta: {args[-1]}, codice {code}; nessuna installazione')
logged([PYTHON,'-u',str(TUNE)])
V2=ROOT/'tuning_v2'
for phase in ('score_validation','calibrate','score_testing','export'):
 logged([PYTHON,'-u',str(V2/'worker.py'),phase])


In [ ]:
V2=ROOT/'tuning_v2'
assert (V2/'export/nira_ehi_nira.tflite').is_file(), 'Nuovo candidato non esportato: nessuna installazione'
CHECK=Path('/content/nyra_exact_stream_v2.py')
CHECK.write_text("\"\"\"Evaluate candidate using the production-style pyopen_wakeword TFLite streaming backend.\nThis is a synthetic compatibility/recall test, not a physical-room accuracy test.\n\"\"\"\nfrom pathlib import Path\nimport sys,json,importlib.util,time,collections,hashlib\nimport numpy as np\nfrom scipy.io.wavfile import read\nsys.path.insert(0,'/content/nyra_pyoww_source')\nfrom pyopen_wakeword import OpenWakeWord,OpenWakeWordFeatures\nROOT=Path('/content/nyra_oww_v1/oww-it-v1-001/tuning_v2');BACKUP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001/tuning_v2')\nspec=importlib.util.spec_from_file_location('worker','/content/nyra_oww_worker.py');w=importlib.util.module_from_spec(spec);spec.loader.exec_module(w)\nold=w.old_worker();head=OpenWakeWord.from_model(ROOT/'export/nira_ehi_nira.tflite');front=OpenWakeWordFeatures.from_builtin();rows=w.load(ROOT/'dataset.json')['rows']\nassert w.sha(ROOT/'export/nira_ehi_nira.tflite')==w.load(ROOT/'export/evaluation.json')['model_sha256']\nrng=np.random.default_rng(91837);selected=[]\nfor phrase in w.PHRASES:\n candidates=[r for r in rows if r['split']=='testing' and r['kind']=='positive' and r['phrase']==phrase]\n selected.extend(candidates[i] for i in rng.choice(len(candidates),50,replace=False))\n# Use all held-out ambient and hard negatives, not just favorable short clips.\nselected.extend(r for r in rows if r['split']=='testing' and r['kind'] in ('ambient','hard','speech'))\nresults=[];started=time.time()\nfor index,r in enumerate(selected):\n sr,a=read(ROOT/r['path']);assert sr==16000 and a.dtype==np.int16;assert w.sha(ROOT/r['path'])==r['sha256']\n for condition,b in old.evaluation_variants(a,r['kind'],w.SEED+r['id']*7919):\n  front.reset();head.reset();pcm=np.clip(b*32767,-32768,32767).astype('<i2');detections=[];max_score=0.\n  for offset in range(0,len(pcm),1280):\n   chunk=pcm[offset:offset+1280]\n   if len(chunk)<1280:chunk=np.pad(chunk,(0,1280-len(chunk)))\n   timestamp=min((offset+1280)/16000,len(pcm)/16000)\n   for emb in front.process_streaming(chunk.tobytes()):\n    for p in head.process_streaming(emb):\n     assert np.isfinite(p);max_score=max(max_score,float(p))\n     if timestamp>=3.2 and p>.5 and (not detections or timestamp-detections[-1]>=2):detections.append(timestamp)\n  results.append(dict(source_id=r['id'],phrase=r['phrase'],kind=r['kind'],condition=condition,duration=len(pcm)/16000,detections=detections,hit=any(4<=t<=4+r['duration']+1 for t in detections),max_scaled_score=max_score))\n if index%25==0:\n  w.save(ROOT/'exact_stream_partial.json',dict(source_rows_done=index,total=len(selected),cases=len(results),elapsed=time.time()-started));print('EXACT_STREAM',index,len(selected),len(results),flush=True)\npositive=[r for r in results if r['kind']=='positive'];ambient=[r for r in results if r['kind']=='ambient'];hard=[r for r in results if r['kind']=='hard'];hours=sum(r['duration'] for r in ambient)/3600\nrecall={p:sum(r['hit'] for r in positive if r['phrase']==p)/sum(r['phrase']==p for r in positive) for p in w.PHRASES}\nby_condition={p:{c:sum(r['hit'] for r in positive if r['phrase']==p and r['condition']==c)/sum(r['phrase']==p and r['condition']==c for r in positive) for c in sorted({r['condition'] for r in positive})} for p in w.PHRASES}\nreport=dict(model_sha256=w.sha(ROOT/'export/nira_ehi_nira.tflite'),backend_commit='6bc5c5f5c9c71e46a723b6c9277b1d50f2ba13fd',recall=recall,recall_by_condition=by_condition,ambient_hours=hours,faph=sum(len(r['detections']) for r in ambient)/hours,hard_negative_fpr=sum(bool(r['detections']) for r in hard)/len(hard),all_audio_synthetic=True,human_test=False,thresholds_selected_on='batch_validation_only',cases=len(results),elapsed_seconds=time.time()-started)\nreport['synthetic_positive_gate_passed']=all(x>=.97 for x in recall.values()) and all(min(v.values())>=.9 for v in by_condition.values())\nw.save(ROOT/'export/exact_stream_evaluation.json',report);w.save(BACKUP/'export/exact_stream_evaluation.json',report);w.save(ROOT/'exact_stream_cases.json',results)\nprint(json.dumps(report,indent=2),flush=True)\nassert report['synthetic_positive_gate_passed'],'Exact streaming positive recall failed: do not deploy. Revisit features/calibration with validation only.'\n")
shutil.copy2(CHECK,BKP/CHECK.name)
logged([PYTHON,'-u',str(CHECK)])


In [ ]:
from pathlib import Path
import subprocess
P=Path('/content/nyra_diagnose_stream_v2.py')
P.write_text("\"\"\"Small validation-only streaming diagnostic. Never a deployable export.\"\"\"\nfrom pathlib import Path\nimport sys,importlib.util,json,time\nimport numpy as np\nfrom scipy.io.wavfile import read\nROOT=Path('/content/nyra_oww_v1/oww-it-v1-001/tuning_v2');OUT=ROOT/'diagnostic_only';OUT.mkdir(exist_ok=True)\nBKP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001/tuning_v2/diagnostic_only');BKP.mkdir(exist_ok=True)\nspec=importlib.util.spec_from_file_location('w',ROOT/'worker.py');w=importlib.util.module_from_spec(spec);spec.loader.exec_module(w)\ntf=w.tf_init()\nfor phrase in w.PHRASES:\n path=OUT/(phrase+'.tflite')\n if not path.exists():\n  head=tf.keras.models.load_model(ROOT/'models'/phrase/'best.keras',compile=False);inp=tf.keras.Input(batch_shape=(1,16,96));m=tf.keras.Model(inp,head(inp));m.export(str(OUT/phrase));conv=tf.lite.TFLiteConverter.from_saved_model(str(OUT/phrase));conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS];path.write_bytes(conv.convert())\nsys.path.insert(0,'/content/nyra_pyoww_source')\nfrom pyopen_wakeword import OpenWakeWord,OpenWakeWordFeatures\nheads={p:OpenWakeWord.from_model(OUT/(p+'.tflite')) for p in w.PHRASES};front=OpenWakeWordFeatures.from_builtin();old=w.old_worker()\nrows=w.load(ROOT/'dataset.json')['rows'];rng=np.random.default_rng(100347);selected=[]\nfor phrase in w.PHRASES:\n a=[r for r in rows if r['split']=='validation' and r['phrase']==phrase and r['kind']=='positive'];selected.extend(a[i] for i in rng.choice(len(a),5,replace=False))\nfor kind,n in (('hard',30),('speech',30),('ambient',5)):\n a=[r for r in rows if r['split']=='validation' and r['kind']==kind];selected.extend(a[i] for i in rng.choice(len(a),n,replace=False))\nresults=[];started=time.time()\nfor index,r in enumerate(selected):\n sr,a=read(r['path']);assert sr==16000 and w.sha(r['path'])==r['sha256']\n for condition,b in old.evaluation_variants(a,r['kind'],w.SEED+r['id']*7919):\n  front.reset()\n  for h in heads.values():h.reset()\n  pcm=np.clip(b*32767,-32768,32767).astype('<i2');probs={p:[] for p in heads};times=[]\n  for offset in range(0,len(pcm),1280):\n   chunk=pcm[offset:offset+1280]\n   if len(chunk)<1280:chunk=np.pad(chunk,(0,1280-len(chunk)))\n   timestamp=min((offset+1280)/16000,len(pcm)/16000)\n   for emb in front.process_streaming(chunk.tobytes()):\n    byhead={p:list(h.process_streaming(emb)) for p,h in heads.items()};n=len(next(iter(byhead.values())));assert all(len(v)==n for v in byhead.values())\n    times.extend([timestamp]*n)\n    for p,v in byhead.items():probs[p].extend(float(x) for x in v)\n  assert all(np.isfinite(v).all() for v in probs.values())\n  results.append(dict(source_id=r['id'],kind=r['kind'],phrase=r['phrase'],condition=condition,duration=len(pcm)/16000,start=4.,end=4.+r['duration'],times=times,probs=probs))\n print('STREAM_DIAGNOSTIC',index+1,len(selected),'cases',len(results),'seconds',round(time.time()-started,1),flush=True)\nreport={'scope':'small fixed validation-only diagnostic; not a final accuracy estimate','cases':len(results),'models':{}}\nfor phrase in heads:\n summary=[]\n for cutoff in (.5,.6,.7,.8,.9,.95,.99):\n  positives=[r for r in results if r['kind']=='positive' and r['phrase']==phrase];negative=[r for r in results if r['kind']!='positive'];hits=0;early=0;fp=0\n  for r in positives:\n   t=np.asarray(r['times']);p=np.asarray(r['probs'][phrase]);hits+=int(np.any((t>=4)&(t<=r['end']+1)&(p>cutoff)));early+=int(np.any((t>=3.2)&(t<4)&(p>cutoff)))\n  for r in negative:\n   t=np.asarray(r['times']);p=np.asarray(r['probs'][phrase]);fp+=int(np.any((t>=3.2)&(p>cutoff)))\n  summary.append(dict(cutoff=cutoff,positive_signal_rate=hits/len(positives),early_signal_rate=early/len(positives),negative_clip_trigger_rate=fp/len(negative)))\n report['models'][phrase]=summary\nw.save(OUT/'streaming_diagnostic.json',report);w.save(BKP/'streaming_diagnostic.json',report);w.save(OUT/'streaming_cases.json',results)\nprint(json.dumps(report,indent=2),flush=True)\n")
with subprocess.Popen([PYTHON,'-u',str(P)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
 for line in proc.stdout: print(line,end='',flush=True)
 assert proc.wait()==0, 'Diagnostica streaming non conclusa; nessuna installazione'


In [ ]:
from pathlib import Path
import subprocess, shutil, time, queue, threading, os, signal
V3ROOT=Path('/content/nyra_oww_v1/oww-it-v1-001/stream_v3'); V3ROOT.mkdir(exist_ok=True)
V3BKP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001/stream_v3'); V3BKP.mkdir(exist_ok=True)
V3WORKER=V3ROOT/'controller.py'
V3WORKER.write_text("\"\"\"Resumable exact-stream training and guarded validation/test controller. Synthetic only.\"\"\"\nfrom pathlib import Path\nimport sys,os,json,time,hashlib,shutil,importlib.util,fcntl,multiprocessing as mp,collections,traceback\nimport numpy as np\nBASE=Path('/content/nyra_oww_v1/oww-it-v1-001');V2=BASE/'tuning_v2';ROOT=BASE/'stream_v3'\nBACKUP=Path('/content/drive/MyDrive/nyra_openwakeword_it/oww-it-v1-001/stream_v3')\nPHRASES=('ehi_nira_it','nira_it');SEED=192311;POLICY='exact-stream-v3-1'\nsys.path.insert(0,'/content/nyra_pyoww_source')\ndef load(p):return json.loads(Path(p).read_text())\ndef save(p,d):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_suffix(p.suffix+'.partial');tmp.write_text(json.dumps(d,indent=2));tmp.replace(p)\ndef sha(p):\n h=hashlib.sha256()\n with open(p,'rb') as f:\n  for b in iter(lambda:f.read(1<<20),b''):h.update(b)\n return h.hexdigest()\ndef status(stage,**kw):\n d=dict(stage=stage,updated=time.time(),**kw);save(ROOT/'progress.json',d);save(BACKUP/'progress.json',d);print(json.dumps(d),flush=True)\ndef worker_module():\n s=importlib.util.spec_from_file_location('approved_worker','/content/nyra_oww_worker.py');w=importlib.util.module_from_spec(s);s.loader.exec_module(w);return w\n_FRONT=None;_OLD=None\ndef init_front():\n global _FRONT,_OLD\n from pyopen_wakeword import OpenWakeWordFeatures\n _FRONT=OpenWakeWordFeatures.from_builtin();_OLD=worker_module().old_worker()\ndef cachepath(item):return ROOT/'stream_features'/item['split']/(hashlib.sha256(json.dumps(item,sort_keys=True).encode()).hexdigest()+'.npz')\ndef extract_one(item):\n p=cachepath(item)\n if p.exists():\n  with np.load(p) as z:assert z['emb'].shape[1]==96 and z['emb'].shape[0]==len(z['times'])\n  return str(p)\n from scipy.io.wavfile import read\n r=item['row'];seed=SEED+r['id']*7919\n if r['kind']=='quiet':a=np.empty(0,np.float32)\n else:\n  sr,a=read(r['path']);assert sr==16000 and a.dtype==np.int16 and sha(r['path'])==r['sha256']\n  a=a.astype(np.float32)/32768\n if item['split']=='training':\n  if r['kind']=='quiet':b=_OLD.continuous_context(a,seed,pre_s=8.,tail_s=0.,floor_dbfs=item['floor'])\n  else:\n   b=_OLD.acoustic(a,item['profile'],seed);b=_OLD.continuous_context(b,seed,pre_s=4.,tail_s=1.,floor_dbfs=item['floor'])\n else:\n  # Exactly reproduce the existing evaluation policy, preserving original source IDs/seeds.\n  original_seed=18473+r['id']*7919\n  if r['kind']=='positive':\n   profile,db=item['condition'].rsplit('__floor',1);i=_OLD.ACOUSTIC_PROFILES.index(profile);j=(-75,-60,-45).index(int(db))\n   b=_OLD.continuous_context(_OLD.acoustic(a,profile,original_seed+i*1301),original_seed+i*7919+j,floor_dbfs=int(db))\n  else:b=_OLD.continuous_context(a,original_seed)\n pcm=np.clip(b*32767,-32768,32767).astype('<i2');_FRONT.reset();emb=[];times=[]\n for offset in range(0,len(pcm),1280):\n  chunk=pcm[offset:offset+1280]\n  if len(chunk)<1280:chunk=np.pad(chunk,(0,1280-len(chunk)))\n  t=min((offset+1280)/16000,len(pcm)/16000)\n  for features in _FRONT.process_streaming(chunk.tobytes()):\n   x=features[0,0];emb.extend(x);times.extend([t]*len(x))\n x=np.asarray(emb,np.float32);t=np.asarray(times,np.float32);assert x.ndim==2 and x.shape[1]==96 and len(x)>=16 and np.isfinite(x).all()\n p.parent.mkdir(parents=True,exist_ok=True)\n with open(p.with_suffix('.partial'),'wb') as f:np.savez_compressed(f,emb=x,times=t,duration=np.array(len(pcm)/16000),meta=np.array(json.dumps(item)))\n p.with_suffix('.partial').replace(p);return str(p)\ndef make_items():\n rows=load(V2/'dataset.json')['rows'];rng=np.random.default_rng(SEED);items=[]\n def add_train(r,i):items.append(dict(policy=POLICY,split='training',condition='training',row=r,profile=('clean','quiet','off_axis_mild','off_axis_strong','reflections','noise_snr10','off_axis_noise')[i%7],floor=(-75,-60,-45)[i%3]))\n # Round robin by family/variant ensures voice and pronunciation coverage.\n for phrase in PHRASES:\n  groups=collections.defaultdict(list)\n  for r in rows:\n   if r['split']=='training' and r['kind']=='positive' and r['phrase']==phrase:groups[(r['family'],r.get('variant'))].append(r)\n  for a in groups.values():rng.shuffle(a)\n  chosen=[]\n  while len(chosen)<192:\n   for key in sorted(groups,key=str):\n    if groups[key]:chosen.append(groups[key].pop())\n    if len(chosen)>=192:break\n  for i,r in enumerate(chosen):add_train(r,i)\n for kind,n in (('hard',512),('continuous',192),('speech',96),('noise',64)):\n  a=[r for r in rows if r['split']=='training' and r['kind']==kind]\n  for i,j in enumerate(rng.choice(len(a),n,replace=False)):add_train(a[j],i)\n for i in range(128):\n  r=dict(id=1000000+i,kind='quiet',phrase='negative',split='training',duration=0.,family='synthetic_floor')\n  add_train(r,i);items[-1]['floor']=(-80,-75,-70,-65,-60,-55,-50,-45)[i%8]\n # Larger validation selection fixed before fine-tuning; held-out test remains unmined.\n old=worker_module().old_worker();conditions=[f'{p}__floor{db}' for p in old.ACOUSTIC_PROFILES for db in (-75,-60,-45)]\n for split,count in (('validation',15),('testing',30)):\n  selected=[]\n  for phrase in PHRASES:\n   a=[r for r in rows if r['split']==split and r['kind']=='positive' and r['phrase']==phrase]\n   selected.extend(a[j] for j in rng.choice(len(a),count,replace=False))\n  for kind,n in ((('hard',120),('speech',120),('ambient',60)) if split=='validation' else (('hard',600),('speech',500),('ambient',240))):\n   a=[r for r in rows if r['split']==split and r['kind']==kind];selected.extend(a[j] for j in rng.choice(len(a),min(n,len(a)),replace=False))\n  for r in selected:\n   for c in conditions if r['kind']=='positive' else ['original']:items.append(dict(policy=POLICY,split=split,condition=c,row=r))\n return items\ndef prepare(items,split):\n selected=[x for x in items if x['split']==split];missing=[x for x in selected if not cachepath(x).exists()]\n status('extract_stream_'+split,total=len(selected),missing=len(missing),workers=2)\n assert shutil.disk_usage(ROOT).free>8*2**30,'Low SSD space: stop and preserve artifacts'\n ctx=mp.get_context('spawn');last=time.monotonic();done=0\n with ctx.Pool(2,initializer=init_front) as pool:\n  it=pool.imap_unordered(extract_one,missing,chunksize=1)\n  while done<len(missing):\n   try:it.next(timeout=30);done+=1;last=time.monotonic()\n   except mp.TimeoutError:\n    status('heartbeat_stream_'+split,completed=done,total_missing=len(missing),seconds_without_completion=round(time.monotonic()-last))\n    if time.monotonic()-last>600:pool.terminate();raise RuntimeError('Streaming extraction made no progress for 10 minutes; cached work preserved')\n   if done and done%20==0:status('extract_stream_'+split,completed=done,total_missing=len(missing))\n save(ROOT/(split+'_features.json'),[dict(item=x,path=str(cachepath(x))) for x in selected]);status('extract_stream_'+split+'_complete',cases=len(selected))\ndef windows(entry):\n with np.load(entry['path']) as z:\n  x=z['emb'];times=z['times'][15:];duration=float(z['duration'])\n return np.ascontiguousarray(np.lib.stride_tricks.sliding_window_view(x,(16,96))[:,0]),times,duration\ndef init_tf():\n import tensorflow as tf\n tf.config.threading.set_intra_op_parallelism_threads(2);tf.config.threading.set_inter_op_parallelism_threads(2)\n for g in tf.config.list_physical_devices('GPU'):tf.config.experimental.set_memory_growth(g,True)\n tf.keras.utils.set_random_seed(SEED);return tf\ndef train():\n tf=init_tf();entries=load(ROOT/'training_features.json');val=load(ROOT/'validation_features.json');original=load(BASE/'features.json')['rows']\n for phrase in PHRASES:\n  dest=ROOT/'models'/phrase;dest.mkdir(parents=True,exist_ok=True);bkp=BACKUP/'models'/phrase;bkp.mkdir(parents=True,exist_ok=True)\n  for name in ('checkpoint.keras','best.keras','state.json','complete.json'):\n   if not (dest/name).exists() and (bkp/name).exists():shutil.copy2(bkp/name,dest/name)\n  if (dest/'complete.json').exists():continue\n  ref=tf.keras.models.load_model(V2/'models'/phrase/'best.keras',compile=False)\n  @tf.function(input_signature=[tf.TensorSpec([None,16,96],tf.float32)])\n  def predict(x):return tf.reshape(ref(x,training=False),[-1])\n  pos=[];neg=[];quiet=[]\n  for e in entries:\n   r=e['item']['row'];x,t,_=windows(e)\n   if r['kind']=='positive':\n    if r['phrase']==phrase:\n     good=(t>=4+r['duration']-.08)&(t<=4+r['duration']+.48);pos.extend(x[good])\n    quiet.extend(x[(t>=3.2)&(t<3.92)][::2])\n   elif r['kind']=='quiet':quiet.extend(x[t>=3.2][::4])\n   else:\n    x=x[t>=3.2];p=predict(x).numpy();idx=np.argsort(p)[-min(8,len(p)):];neg.extend(x[idx]);neg.extend(x[::max(1,len(x)//4)][:4])\n  pos=np.stack(pos);neg=np.stack(neg);quiet=np.stack(quiet)\n  chosen=[r for r in original if r['split']=='training' and (r['kind']!='positive' or r['phrase']==phrase)]\n  ox=np.stack([np.load(BASE/r['feature']) for r in chosen]);oy=np.array([r['kind']=='positive' for r in chosen]);op=np.flatnonzero(oy);on=np.flatnonzero(~oy)\n  vp=[];vn=[]\n  for e in val:\n   r=e['item']['row'];x,t,_=windows(e)\n   if r['kind']=='positive':\n    if r['phrase']==phrase:vp.extend(x[(t>=4+r['duration'])&(t<=4+r['duration']+.48)][::2])\n    vn.extend(x[(t>=3.2)&(t<3.92)][::3])\n   else:vn.extend(x[t>=3.2][::max(1,len(x)//12)])\n  rng=np.random.default_rng(SEED);vp=np.stack(vp);vn=np.stack(vn);n=min(2500,len(vp),len(vn));vx=np.concatenate([vp[rng.choice(len(vp),n,False)],vn[rng.choice(len(vn),n,False)]]);vy=np.r_[np.ones(n),np.zeros(n)].astype(np.float32)[:,None]\n  if (dest/'checkpoint.keras').exists():m=tf.keras.models.load_model(dest/'checkpoint.keras');st=load(dest/'state.json')\n  else:m=tf.keras.models.load_model(V2/'models'/phrase/'best.keras');m.optimizer.learning_rate.assign(1e-4);st=dict(step=0,best_loss=1e9,source='V2 best',exact_stream_positive_windows=len(pos),exact_stream_negative_windows=len(neg),quiet_windows=len(quiet))\n  for step in range(st['step'],4000):\n   rng=np.random.default_rng(SEED+step);xp=np.concatenate([ox[rng.choice(op,64)],pos[rng.choice(len(pos),64)]]);xn=np.concatenate([ox[rng.choice(on,32)],neg[rng.choice(len(neg),64)],quiet[rng.choice(len(quiet),32)]])\n   loss=float(m.train_on_batch(np.concatenate([xp,xn]),np.r_[np.ones(128),np.zeros(128)].astype(np.float32)))\n   if (step+1)%250==0:\n    vl=float(tf.reduce_mean(tf.keras.losses.binary_crossentropy(vy,m(vx,training=False))));st['step']=step+1\n    if vl<st['best_loss']:st['best_loss']=vl;m.save(dest/'best.keras');shutil.copy2(dest/'best.keras',bkp/'best.keras')\n    m.save(dest/'checkpoint.keras');save(dest/'state.json',st)\n    for name in ('checkpoint.keras','state.json'):shutil.copy2(dest/name,bkp/name)\n    status('train_exact',phrase=phrase,step=step+1,loss=loss,validation_loss=vl)\n  save(dest/'complete.json',st);shutil.copy2(dest/'complete.json',bkp/'complete.json');del m,ref,ox,pos,neg,quiet,vx;tf.keras.backend.clear_session()\n status('train_exact_complete')\ndef score(split):\n tf=init_tf();models=[tf.keras.models.load_model(ROOT/'models'/p/'best.keras',compile=False) for p in PHRASES]\n @tf.function(input_signature=[tf.TensorSpec([None,16,96],tf.float32)])\n def pred(x):return tf.concat([m(x,training=False) for m in models],axis=1)\n entries=load(ROOT/(split+'_features.json'));results=[]\n for i,e in enumerate(entries):\n  x,t,d=windows(e);p=pred(x).numpy();r=e['item']['row'];results.append(dict(id=r['id'],kind=r['kind'],phrase=r['phrase'],condition=e['item']['condition'],duration=d,start=4.,end=4.+r['duration'],times=t.tolist(),probs={s:p[:,j].tolist() for j,s in enumerate(PHRASES)}))\n  if i%100==0:status('score_exact_'+split,done=i,total=len(entries))\n import gzip\n with gzip.open(ROOT/(split+'_scores.json.gz'),'wt') as f:json.dump(results,f)\n shutil.copy2(ROOT/(split+'_scores.json.gz'),BACKUP/(split+'_scores.json.gz'));status('score_exact_'+split+'_complete',cases=len(results))\ndef event_times(r,cuts,n):\n p=np.maximum(np.asarray(r['probs'][PHRASES[0]])/cuts[0],np.asarray(r['probs'][PHRASES[1]])/cuts[1]);t=np.asarray(r['times'])\n # Each phrase must independently exceed its cutoff on n consecutive frames.\n if n>1:\n  p=np.maximum(*[np.min(np.lib.stride_tricks.sliding_window_view(np.asarray(r['probs'][name])/cut,n),axis=1) for name,cut in zip(PHRASES,cuts)]);t=t[n-1:]\n candidates=t[(t>=3.2)&(p>1.)];events=[]\n for ti in candidates:\n  if not events or ti-events[-1]>=2.:events.append(float(ti))\n return events\ndef metrics(rows,cuts,n):\n hit=collections.Counter();total=collections.Counter();conditions=collections.defaultdict(lambda:[0,0]);ambient_events=0;hours=0.;hardhits=hardtotal=0;early=0;lat=[]\n for r in rows:\n  ev=event_times(r,cuts,n)\n  if r['kind']=='positive':\n   good=[t for t in ev if r['start']<=t<=r['end']+1.];ok=bool(good);hit[r['phrase']]+=ok;total[r['phrase']]+=1;c=conditions[(r['phrase'],r['condition'])];c[0]+=ok;c[1]+=1;early+=any(t<r['start'] for t in ev)\n   if good:lat.append(good[0]-r['end'])\n  elif r['kind']=='ambient':ambient_events+=len(ev);hours+=r['duration']/3600\n  elif r['kind']=='hard':hardhits+=bool(ev);hardtotal+=1\n return dict(cutoffs=dict(zip(PHRASES,cuts)),frames=n,recall={p:hit[p]/total[p] for p in PHRASES},min_condition=min(a/b for a,b in conditions.values()),faph=ambient_events/hours,ambient_events=ambient_events,ambient_hours=hours,hard_fpr=hardhits/hardtotal,preword_triggers=early,latency_p90_s=float(np.percentile(lat,90)) if lat else None)\ndef calibrate():\n import gzip\n with gzip.open(ROOT/'validation_scores.json.gz','rt') as f:rows=json.load(f)\n grid=[]\n for n in (1,2,3):\n  for a in (.4,.5,.6,.7,.8,.85,.9,.95,.975,.99):\n   for b in (.4,.5,.6,.7,.8,.85,.9,.95,.975,.99):grid.append(metrics(rows,(a,b),n))\n good=[r for r in grid if min(r['recall'].values())>=.99 and r['min_condition']>=.9 and r['faph']<=2 and r['hard_fpr']<=.05 and r['preword_triggers']==0 and r['latency_p90_s']<=.8]\n selected=min(good,key=lambda r:(r['faph'],r['hard_fpr'],r['latency_p90_s'])) if good else None\n save(ROOT/'operating_points.json',dict(selected=selected,grid=grid));shutil.copy2(ROOT/'operating_points.json',BACKUP/'operating_points.json')\n status('calibrate_exact_complete',selected=selected,best_positive=min(grid,key=lambda r:(-min(r['recall'].values()),r['faph'],r['hard_fpr'])))\n assert selected,'Exact-stream validation gate failed. Preserve weights and do not install.'\ndef export():\n import gzip\n op=load(ROOT/'operating_points.json')['selected'];assert op\n with gzip.open(ROOT/'testing_scores.json.gz','rt') as f:rows=json.load(f)\n cuts=[op['cutoffs'][p] for p in PHRASES];report=metrics(rows,cuts,op['frames']);report.update(all_audio_synthetic=True,human_test=False,threshold_selection='exact_stream_validation_only',backend='pyopen_wakeword',backend_commit='6bc5c5f5c9c71e46a723b6c9277b1d50f2ba13fd')\n save(ROOT/'heldout_report.json',report);shutil.copy2(ROOT/'heldout_report.json',BACKUP/'heldout_report.json')\n assert min(report['recall'].values())>=.97 and report['min_condition']>=.85 and report['faph']<=2 and report['hard_fpr']<=.05,'Held-out exact streaming quality gate failed; do not install.'\n tf=init_tf();n=op['frames'];inp=tf.keras.Input(batch_shape=(1,16+n-1,96));heads=[]\n for phrase,cut in zip(PHRASES,cuts):\n  m=tf.keras.models.load_model(ROOT/'models'/phrase/'best.keras',compile=False);scores=[]\n  for i in range(n):scores.append(m(inp[:,i:i+16,:]))\n  s=scores[0] if n==1 else tf.keras.layers.Minimum()(scores);heads.append(tf.keras.layers.Rescaling(1/(2*cut))(s))\n combo=tf.keras.Model(inp,tf.keras.layers.Maximum()(heads));out=ROOT/'export';out.mkdir(exist_ok=True);combo.export(str(out/'saved_model'));c=tf.lite.TFLiteConverter.from_saved_model(str(out/'saved_model'));c.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS];path=out/'nira_ehi_nira.tflite';path.write_bytes(c.convert())\n from pyopen_wakeword import OpenWakeWord\n model=OpenWakeWord.from_model(path);rng=np.random.default_rng(SEED)\n for i in range(20):\n  x=rng.normal(size=(1,16+n-1,96)).astype(np.float32);model.reset();actual=list(model.process_streaming(x.reshape(1,1,16+n-1,96)));expected=float(combo(x).numpy()[0,0]);assert len(actual)==1 and np.allclose(actual[0],expected,atol=1e-4),('TFLite C head mismatch',actual,expected)\n report.update(model_sha256=sha(path),input_frames=16+n-1,tflite_c_numerical_check='20 inputs passed',deployment_ready=True)\n save(out/'evaluation.json',report);shutil.copytree(out,BACKUP/'export',dirs_exist_ok=True);status('ready_for_device_validation',report=report)\ndef main():\n ROOT.mkdir(parents=True,exist_ok=True);BACKUP.mkdir(parents=True,exist_ok=True);lock=open(ROOT/'controller.lock','w');fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n phase=sys.argv[1]\n if phase=='prepare_plan':\n  if not (ROOT/'plan.json').exists():save(ROOT/'plan.json',make_items())\n  shutil.copy2(ROOT/'plan.json',BACKUP/'plan.json');status('plan_complete',counts=dict(collections.Counter(x['split'] for x in load(ROOT/'plan.json'))))\n elif phase=='smoke':\n  items=load(ROOT/'plan.json');chosen=[]\n  for kind in ('positive','hard','continuous','quiet'):\n   chosen.append(next(x for x in items if x['split']=='training' and x['row']['kind']==kind))\n  chosen.append(next(x for x in items if x['split']=='validation'))\n  init_front()\n  for item in chosen:\n   path=extract_one(item);x,t,d=windows(dict(path=path));assert x.shape[1:]==(16,96) and np.any(t>=3.2)\n   if item['row']['kind']=='positive':assert np.any((t>=4+item['row']['duration'])&(t<=4+item['row']['duration']+.48))\n  status('smoke_passed',cases=len(chosen),backend='exact TFLite C frontend')\n elif phase.startswith('prepare_'):prepare(load(ROOT/'plan.json'),phase.removeprefix('prepare_'))\n elif phase=='train':train()\n elif phase.startswith('score_'):score(phase.removeprefix('score_'))\n elif phase=='calibrate':calibrate()\n elif phase=='export':export()\n else:raise ValueError(phase)\nif __name__=='__main__':\n try:main()\n except Exception as e:\n  if ROOT.exists() and BACKUP.exists():status('failed_preserved',phase=sys.argv[1],error=type(e).__name__+': '+str(e))\n  raise\n")
shutil.copy2(V3WORKER,V3BKP/'controller.py')
def controlled(phase):
 print('CONTROLLER_STAGE',phase,flush=True)
 log=(V3ROOT/'controller.log').open('a')
 with subprocess.Popen([PYTHON,'-u',str(V3WORKER),phase],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True) as proc:
  q=queue.Queue()
  def reader():
   for line in proc.stdout:q.put(line)
   q.put(None)
  threading.Thread(target=reader,daemon=True).start(); last=time.monotonic()
  while True:
   try:line=q.get(timeout=30)
   except queue.Empty:
    print('CONTROLLER_HEARTBEAT',phase,'pid',proc.pid,'silence_s',round(time.monotonic()-last),flush=True)
    if time.monotonic()-last>600:
     os.killpg(proc.pid,signal.SIGTERM);raise RuntimeError('Nessun avanzamento per 10 minuti: fase fermata, cache e checkpoint conservati')
    continue
   if line is None:break
   last=time.monotonic();log.write(line);log.flush();print(line,end='',flush=True)
  code=proc.wait()
 log.close()
 shutil.copy2(V3ROOT/'controller.log',V3BKP/'controller.log')
 if code:raise RuntimeError(f'Fase {phase} interrotta con codice {code}; installazione non autorizzata dal controllo qualità')
for phase in ('prepare_plan','smoke','prepare_training','prepare_validation','train','score_validation','calibrate','prepare_testing','score_testing','export'):
 controlled(phase)
print('READY_FOR_DEVICE_VALIDATION: modello verificato; installazione richiede controllo configurazioni e log dei due speaker.')
